# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PaNavar369/Internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Finding 1 — CTR changes substantially by position

The paper reports that click-through rate is not flat across search positions and declines as pages move further down the results. My methodology question is: how exactly was the outcome defined for this finding, and was CTR measured over a consistent observation window for the different position groups? I would also check whether the validation design supports the claim across different pages and whether differences in query mix or SERP features could partly explain the observed CTR differences. This would help determine how broadly the finding should be interpreted.

Finding 2 — Freshness is associated with search performance

The paper reports a relationship between content freshness and performance, including a freshness-related effect in its analysis. My methodology question is: how was freshness defined and how was the performance outcome measured relative to the update date? I would also check whether the validation design separates information available before the outcome from information observed afterward. If the analysis uses historical comparisons rather than a future holdout, the result should be interpreted as an observed association rather than evidence that updating content causes the measured performance change.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Before/after validation

In Week 5, I evaluated the Decision Tree using a random 80/20 split and measured an accuracy of 60.15% on 6,000 test observations. For this validation audit, I changed the split to group observations by client_id, so clients in the test set were not present in the training set. Under this more conservative split, the model achieved 52.21% accuracy on 6,163 test observations.

The measured accuracy decreased by approximately 7.93 percentage points under the client-grouped split. This suggests that the random split may give a more optimistic estimate of performance when observations from the same clients can be represented across training and testing. The client-grouped result provides a more conservative view of how the model may generalise to unseen clients.

In [6]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

features = [
    "days_since_last_update",
    "ctr",
    "avg_position",
    "impressions_90d"
]

target = "trend_direction"

model_df = df[
    ["client_id", "content_id"] + features + [target]
].dropna(
    subset=features + [target]
).copy()

X = model_df[features]
y = model_df[target]
groups = model_df["client_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

train_clients = set(model_df.iloc[train_idx]["client_id"])
test_clients = set(model_df.iloc[test_idx]["client_id"])

model = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

grouped_accuracy = accuracy_score(y_test, pred)

print("Client-grouped test rows:", len(X_test))
print("Client-grouped accuracy:", grouped_accuracy)
print("Client overlap:", len(train_clients.intersection(test_clients)))

Client-grouped test rows: 6163
Client-grouped accuracy: 0.5221483043972092
Client overlap: 0


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


eakage audit

I checked the final feature set used by my Decision Tree for information that could leak the target into the model. The target is trend_direction, so this field must not be used as an input feature. I also excluded trend_pct because it describes the observed trend outcome and could provide information about the target.

The four model features are days_since_last_update, ctr, avg_position, and impressions_90d. These features describe content freshness and observed search performance, but their timing needs to be considered when using the model for decision support. In particular, CTR, average position, and impressions should represent information available during the defined observation period rather than information collected after the target outcome.

Based on the feature list used in the model, trend_direction and trend_pct are not included as predictors. I therefore found no direct target-field leakage in the final feature set. I would still treat the timing of the search-performance features as an important limitation when interpreting the model.

In [7]:
# Final feature set used by the Week-5 Decision Tree

features = [
    "days_since_last_update",
    "ctr",
    "avg_position",
    "impressions_90d"
]

target = "trend_direction"

leakage_candidates = [
    "trend_direction",
    "trend_pct"
]

print("Final model features:")
for feature in features:
    print("-", feature)

print("\nTarget:")
print("-", target)

print("\nLeakage candidate check:")

for column in leakage_candidates:
    if column in features:
        print(column, "-> LEAKAGE RISK: used as a feature")
    else:
        print(column, "-> NOT USED as a feature")

print("\nDirect target leakage check:")

if target not in features:
    print("PASS: target is not included in model features.")
else:
    print("FAIL: target is included in model features.")

Final model features:
- days_since_last_update
- ctr
- avg_position
- impressions_90d

Target:
- trend_direction

Leakage candidate check:
trend_direction -> NOT USED as a feature
trend_pct -> NOT USED as a feature

Direct target leakage check:
PASS: target is not included in model features.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Original claim

"The machine learning model can predict which pages will improve their search rankings."

Rewritten claim

"The Decision Tree showed an observed and measured accuracy of 60.15% under the Week-5 random split and 52.21% under the client-grouped validation. The lower result under the grouped split suggests that validation design affects the measured performance. These results are directional and should be treated as decision-support, rather than evidence that the selected signals cause ranking changes or that the model can reliably predict future search rankings."

In [8]:
results = {
    "Week-5 random split accuracy": 0.6015,
    "Week-6 client-grouped accuracy": 0.5221483043972092
}

for name, score in results.items():
    print(f"{name}: {score:.4f} ({score:.2%})")

Week-5 random split accuracy: 0.6015 (60.15%)
Week-6 client-grouped accuracy: 0.5221 (52.21%)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.